# PCIe REFCLK SSC deviation: step-by-step walkthrough

This notebook shows every step used by `ssc_analysis.py` to measure the **Spread-Spectrum Clocking (SSC) deviation** of a PCIe 100 MHz REFCLK. It uses a synthetic waveform from `generate_waveform.py`, so the true values are known and each step can be checked.

**Flow**

1. Generate a differential REFCLK waveform with known SSC and jitter, then save and reload it as CSV
2. Detect rising 0 V crossings (hysteresis + linear interpolation)
3. Convert periods to instantaneous frequency
4. Low-pass filter the frequency profile to remove jitter
5. Find SSC cycles and the modulation frequency
6. Compute the SSC deviation from per-cycle extremes
7. Fit a triangle to the raw data (avoids the filter bias)
8. Measure SSC slew rate (df/dt)
9. Run the full analysis in one call, check limits, and run sensitivity experiments

The key definition:

$$\delta_{SSC} = \frac{f_{max} - f_{min}}{f_{max}}$$

PCIe uses **down-spread**: $f_{max} \approx 100$ MHz and $f_{min} \approx 100\,\text{MHz}\,(1-\delta)$, with $\delta \le 0.5\,\%$ (5000 ppm) and a modulation frequency of 30–33 kHz.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

import generate_waveform as gen
import ssc_analysis as ssc

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

## 1. Generate an example waveform

Change these parameters to explore other cases. The generator:

* builds the instantaneous frequency $f(t) = f_{nom}(1+\text{offset})(1+d(t))$, where $d(t)$ is the SSC profile;
* integrates phase and places the ideal edges every ½ cycle;
* adds jitter to each edge as a time displacement: random jitter (RJ, Gaussian) and periodic jitter (PJ, sinusoidal);
* draws erf-shaped edges with the chosen 20–80 % rise time and adds vertical noise.

In [ ]:
params = dict(
    f_nom=100e6,        # Hz
    ssc_ppm=5000,       # p-p SSC deviation (5000 ppm = 0.5 %)
    ssc_freq=31.5e3,    # Hz, modulation frequency
    spread="down",      # down | center | up
    shape="triangle",   # triangle | sine
    offset_ppm=50,      # static oscillator offset
    rj_rms=2e-12,       # s, random jitter per edge
    pj_amp=5e-12,       # s, periodic jitter peak
    pj_freq=5e6,        # Hz
    duration=100e-6,    # s, ~3 SSC periods
    fs=20e9,            # Sa/s
    amplitude=0.7,      # V, differential peak
    rise_time=150e-12,  # s, 20-80 %
    noise_rms=2e-3,     # V
    seed=42,
)
F_NOM = params["f_nom"]

t_gen, v_gen = gen.generate(**params)
print(f"{t_gen.size:,} samples, {t_gen[-1]*1e6:.1f} us @ {params['fs']/1e9:.0f} GSa/s")

### Save to CSV and reload

This is the same format the analyzer expects for a scope capture: column 1 = time [s], column 2 = differential voltage [V]. Time is stored with 12 significant digits. With fewer digits, the rounding of the time values would itself add jitter.

In [ ]:
csv_path = "example_refclk.csv"
np.savetxt(csv_path, np.column_stack((t_gen, v_gen)), delimiter=",",
           fmt=["%.12e", "%.6e"], header="time_s,vdiff_V", comments="")

t, v = ssc.load_waveform(csv_path)
print(f"Loaded {t.size:,} samples, dt = {np.median(np.diff(t))*1e12:.1f} ps")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
m = t < 50e-9
a1.plot(t[m] * 1e9, v[m])
a1.set(xlabel="time [ns]", ylabel="Vdiff [V]", title="First 5 clock cycles")

# Zoom on one rising edge to see the individual samples
k0 = np.flatnonzero((v[:-1] < 0) & (v[1:] >= 0))[2]
z = slice(k0 - 8, k0 + 9)
a2.plot(t[z] * 1e12, v[z], "o-")
a2.axhline(0, color="k", lw=0.8)
a2.set(xlabel="time [ps]", ylabel="Vdiff [V]", title="One rising edge (samples)")
plt.tight_layout()

## 2. Rising-edge detection

* **Threshold:** 0 V differential, the reference point PCIe uses for timing.
* **Hysteresis:** a crossing is accepted only after the signal was below $-h/2$ and then rises above $+h/2$. Noise near 0 V therefore cannot create extra edges. The default is $h$ = 20 % of the 5–95 % swing.
* **Linear interpolation** between the two samples around the crossing:

$$t_c = t_k + \frac{V_{th} - v_k}{v_{k+1}-v_k}(t_{k+1}-t_k)$$

The timing error per crossing is about $\sigma_v / \text{slew}$. For example, 2 mV of noise on a ~5 V/ns edge gives about 0.4 ps.

Only **rising** edges are used, so duty-cycle distortion does not affect the result.

In [ ]:
edges = ssc.rising_edges(t, v, threshold=0.0)
print(f"Rising edges: {edges.size}")

lo, hi = np.percentile(v, [5, 95])
h = 0.2 * (hi - lo)
m = t < 30e-9
plt.plot(t[m] * 1e9, v[m], label="Vdiff")
plt.axhspan(-h / 2, h / 2, color="orange", alpha=0.2, label="hysteresis band")
e = edges[edges < 30e-9]
plt.plot(e * 1e9, np.zeros_like(e), "rv", ms=10, label="interpolated rising crossings")
plt.xlabel("time [ns]"); plt.ylabel("Vdiff [V]"); plt.legend(loc="lower right")

## 3. Periods → instantaneous frequency

The SSC rate (~30 kHz) is about 3000× slower than the clock, so the frequency is effectively constant within one period:

$$f_n = \frac{1}{t_{n+1}-t_n}, \quad \text{at } \tfrac12(t_n+t_{n+1})$$

Each sample also contains **period jitter**. With independent edge jitter $\sigma_{RJ}$, the per-cycle frequency noise is

$$\frac{\sigma_f}{f} \approx \sqrt{2}\,\sigma_{RJ}\,f_0$$

This is ~283 ppm rms for 2 ps RJ at 100 MHz, which is clearly visible against the 5000 ppm SSC.

In [ ]:
t_f, f_raw = ssc.frequency_vs_time(edges)
to_ppm = lambda f: (f / F_NOM - 1) * 1e6

plt.plot(t_f * 1e6, to_ppm(f_raw), ".", ms=1.5, alpha=0.5)
plt.xlabel("time [us]"); plt.ylabel("f vs 100 MHz [ppm]")
plt.title("Per-cycle frequency (raw): SSC triangle + period jitter")

expected = np.sqrt(2) * params["rj_rms"] * F_NOM * 1e6
print(f"Expected per-cycle noise from RJ alone: ~{expected:.0f} ppm rms")

## 4. Low-pass filter the frequency profile

SSC energy is at $f_m$ and its harmonics (a triangle has odd harmonics that fall off as $1/n^2$). Period jitter is broadband and rises with frequency. A **zero-phase Butterworth low-pass** (`sosfiltfilt`, default 2 MHz, 2nd order) therefore separates the two.

Before filtering, the data is resampled onto a uniform grid (step = median period). Filter start-up transients are trimmed from both ends.

**Choosing the cutoff is a trade-off:**

* too low → the triangle corners get rounded and the deviation is **under-estimated**. The relative loss is roughly $\frac{4}{\pi^2 N}$ with $N = f_c/f_m$, which is ~0.6 % at 2 MHz;
* too high → more jitter passes through and biases the max/min **upward**.

In [ ]:
LPF = 2e6
tu, ff = ssc.lowpass_profile(t_f, f_raw, LPF, order=2)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
a1.plot(t_f * 1e6, to_ppm(f_raw), ".", ms=1, alpha=0.3, label="raw")
a1.plot(tu * 1e6, to_ppm(ff), "r", lw=1.5, label=f"low-pass {LPF/1e6:g} MHz")
a1.set(xlabel="time [us]", ylabel="ppm", title="Filtered SSC profile"); a1.legend()

fs_prof = 1 / np.median(np.diff(t_f))
sos = signal.butter(2, LPF, fs=fs_prof, output="sos")
w, H = signal.sosfreqz(sos, worN=4096, fs=fs_prof)
a2.semilogx(w[1:] / 1e6, 20 * np.log10(np.abs(H[1:]) ** 2), label="|H|^2 (filtfilt = applied twice)")
a2.axvline(params["ssc_freq"] / 1e6, color="g", ls="--", label="SSC fundamental")
a2.set(xlabel="frequency [MHz]", ylabel="dB", ylim=(-80, 5), title="Effective filter response"); a2.legend()
plt.tight_layout()

## 5. SSC cycles and modulation frequency

The filtered profile is compared with its mid-level, using a 10 % hysteresis band. Each upward crossing marks the start of a new SSC cycle. With $N$ crossings:

$$f_m = \frac{N-1}{t_{last}-t_{first}}$$

If fewer than 2 crossings are found, the code falls back to the FFT peak.

In [ ]:
fm, trans = ssc.modulation_cycles(tu, ff)
print(f"Modulation frequency: {fm/1e3:.3f} kHz (generated {params['ssc_freq']/1e3:.3f} kHz)")
print(f"Complete SSC cycles : {trans.size - 1}")

## 6. SSC deviation from per-cycle extremes

In each complete cycle, the code takes the max and min of the filtered profile and then averages them over all cycles. Averaging reduces the upward bias that residual noise puts on a single global extreme. The cycle-to-cycle spread indicates how repeatable the measurement is.

In [ ]:
cyc = list(zip(trans[:-1], trans[1:]))
maxima = np.array([ff[a:b].max() for a, b in cyc])
minima = np.array([ff[a:b].min() for a, b in cyc])
f_max, f_min = maxima.mean(), minima.mean()
spread = (f_max - f_min) / f_max * 1e6

for i, (mx, mn) in enumerate(zip(maxima, minima)):
    print(f"cycle {i}: max {to_ppm(mx):+7.0f} ppm, min {to_ppm(mn):+7.0f} ppm, spread {(mx-mn)/mx*1e6:6.0f} ppm")
print(f"\nSSC deviation = {spread:.0f} ppm ({spread/1e4:.4f} %)   [generated: {params['ssc_ppm']} ppm]")
print(f"f_max = {to_ppm(f_max):+.0f} ppm, f_min = {to_ppm(f_min):+.0f} ppm vs 100 MHz (generated offset {params['offset_ppm']} ppm)")

plt.plot(tu * 1e6, to_ppm(ff), "r")
for a, b in cyc:
    plt.plot(tu[a + np.argmax(ff[a:b])] * 1e6, to_ppm(ff[a:b].max()), "g^")
    plt.plot(tu[a + np.argmin(ff[a:b])] * 1e6, to_ppm(ff[a:b].min()), "mv")
for i in trans:
    plt.axvline(tu[i] * 1e6, color="k", lw=0.5)
plt.xlabel("time [us]"); plt.ylabel("ppm"); plt.title("Per-cycle extremes (cycle boundaries in black)")

## 7. Triangle least-squares fit (no filter)

Model:

$$f(t) = f_{hi} - (f_{hi}-f_{lo})\cdot \mathrm{tri}\big(f_m(t-t_p)\big)$$

The model is fitted to the **raw** per-cycle frequencies and initialized from the filtered estimate. Thousands of points are averaged, so the jitter cancels without any corner-rounding bias. This gives the most accurate result **when the profile really is triangular**.

If the fit residual is clearly larger than the raw per-cycle noise, the profile is not triangular (for example sine or "Hershey-kiss"). In that case, use the filtered result instead.

In [ ]:
inside = (t_f >= tu[0]) & (t_f <= tu[-1])
t_peak = tu[np.argmax(ff[: int(1 / fm / (tu[1] - tu[0]))])]
f_hi, f_lo, fm_fit, rms = ssc.fit_triangle(t_f[inside], f_raw[inside], F_NOM, fm, t_peak, f_max, f_min)

raw_noise = np.std(f_raw[inside] - np.interp(t_f[inside], tu, ff)) / F_NOM * 1e6
print(f"Fit: f_hi {to_ppm(f_hi):+.0f} ppm, f_lo {to_ppm(f_lo):+.0f} ppm, fm {fm_fit/1e3:.3f} kHz")
print(f"Fit SSC deviation = {(f_hi - f_lo)/f_hi*1e6:.0f} ppm")
print(f"Fit residual {rms:.0f} ppm rms vs raw noise {raw_noise:.0f} ppm rms")

## 8. SSC slew rate (df/dt)

The slope of the filtered profile over a 0.5 µs window is computed, and the maximum $|df/dt|$ is reported in ppm/µs. An ideal triangle has slope $2\delta f_m$, which is ≈ 315 ppm/µs for 0.5 % at 31.5 kHz. The commonly cited PCIe limit is 1250 ppm/µs.

In [ ]:
slew = ssc.max_slew_ppm_per_us(tu, ff, F_NOM)
ideal = 2 * params["ssc_ppm"] * params["ssc_freq"] * 1e-6
print(f"max |df/dt| = {slew:.0f} ppm/us (ideal triangle {ideal:.0f} ppm/us)")

## 9. Full analysis in one call

`ssc.analyze()` runs all of the steps above. `ssc.report()` prints the summary, and `ssc.plot()` draws the diagnostic plot. The command-line equivalent is:

```
python ssc_analysis.py example_refclk.csv --plot
```

In [ ]:
r = ssc.analyze(t, v, f_nom=F_NOM, lpf=LPF)
ssc.report(r, F_NOM)
ssc.plot(r, F_NOM)

### Check against commonly cited PCIe limits

Verify these limits against the PCIe Base/CEM spec revision you are testing to.

In [ ]:
spread_fit = (r["fit"][0] - r["fit"][1]) / r["fit"][0] * 1e6 if r["fit"] else r["spread_ppm"]
checks = [
    ("f_max within +/-300 ppm of 100 MHz", abs(r["f_max_vs_nom_ppm"]) <= 300, f"{r['f_max_vs_nom_ppm']:+.0f} ppm"),
    ("SSC deviation <= 5000 ppm (0.5 %)", spread_fit <= 5000 + 1, f"{spread_fit:.0f} ppm"),
    ("Modulation 30-33 kHz", 30e3 <= r["ssc_freq_hz"] <= 33e3, f"{r['ssc_freq_hz']/1e3:.2f} kHz"),
    ("|df/dt| <= 1250 ppm/us", r["max_slew_ppm_per_us"] <= 1250, f"{r['max_slew_ppm_per_us']:.0f} ppm/us"),
]
for name, ok, val in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {name:38s} {val}")

## 10. Experiments

### 10a. Effect of the low-pass cutoff

A low cutoff rounds the triangle corners and under-estimates the deviation. A high cutoff lets jitter through and over-estimates it. The triangle fit does not depend on the cutoff.

In [ ]:
cutoffs = [0.3e6, 0.5e6, 1e6, 2e6, 5e6, 10e6, 20e6]
est = [ssc.analyze(t, v, F_NOM, lpf=c)["spread_ppm"] for c in cutoffs]
plt.semilogx(np.array(cutoffs) / 1e6, est, "o-", label="filtered + per-cycle extremes")
plt.axhline(params["ssc_ppm"], color="k", ls="--", label="true")
plt.axhline((r["fit"][0] - r["fit"][1]) / r["fit"][0] * 1e6, color="g", ls=":", label="triangle fit")
plt.xlabel("LPF cutoff [MHz]"); plt.ylabel("SSC deviation [ppm]"); plt.legend()

### 10b. Effect of random jitter

Shorter captures (40 µs) are used here to keep it fast.

In [ ]:
rjs = [0, 1, 2, 5, 10, 20]
rows = []
for rj in rjs:
    p = dict(params, rj_rms=rj * 1e-12, pj_amp=0, duration=40e-6, seed=1)
    tt, vv = gen.generate(**p)
    rr = ssc.analyze(tt, vv, F_NOM, lpf=LPF)
    fit = (rr["fit"][0] - rr["fit"][1]) / rr["fit"][0] * 1e6 if rr["fit"] else np.nan
    rows.append((rj, rr["raw_freq_std_ppm"], rr["spread_ppm"], fit))
    print(f"RJ {rj:3d} ps: raw noise {rows[-1][1]:6.0f} ppm rms | filtered {rows[-1][2]:6.0f} ppm | fit {fit:6.0f} ppm")

### 10c. Non-triangular profile (sine, center-spread)

The filtered estimate still works. The triangle-fit residual rises above the raw noise, and the report flags that the fit does not apply.

In [ ]:
tt, vv = gen.generate(**dict(params, shape="sine", spread="center", seed=7))
rs = ssc.analyze(tt, vv, F_NOM, lpf=LPF)
ssc.report(rs, F_NOM)

## Summary of assumptions

* Column 2 is the **differential** voltage, and timing is measured at the 0 V crossing.
* There are at least ~3–5 samples per edge (≥ 10–20 GSa/s), so linear interpolation of the crossing is accurate.
* The capture contains **≥ 2 full SSC cycles** (≥ ~100 µs). Shorter captures fall back to global max/min and print a warning.
* SSC is slow compared with the clock ($f_m \ll f_0$), so one period is one sample of the instantaneous frequency.
* Jitter is zero-mean and mostly above the LPF cutoff. Wander below the cutoff cannot be separated from SSC.
* A scope timebase error shifts the absolute frequencies, but not the deviation (a ratio).
* Use the triangle fit only when its residual ≈ the raw per-cycle noise.

Analyze a real capture by pointing `ssc.load_waveform()` (or the CLI) at your scope CSV.